# PC-GAT : Predictive Coding meets Graph Attention Networks

Une architecture bio-inspirée qui combine :

- **Predictive Coding (PC)** : inférence de croyances hiérarchiques par descente de gradient locale
- **Graph Attention Networks (GAT)** : attention sur les voisinages de graphe

## Propriétés clés

| Propriété | Mécanisme |
|---|---|
| **Attention par surprise** | Les nœuds mal prédits reçoivent plus d'attention |
| **Inférence itérative** | Convergence des croyances avant l'apprentissage |
| **Robustesse au bruit** | Les erreurs aléatoires sont absorbées par la dynamique PC |
| **Plausibilité biologique** | Règles locales, pas de backprop global |
| **Détection d'anomalies native** | Les nœuds atypiques génèrent des erreurs `ε` élevées |

---

## Fondements théoriques

### 1. Prédiction top-down sur le graphe

Chaque nœud $i$ à la couche $l$ génère une prédiction en agrégeant les représentations voisines de la couche supérieure :

$$\hat{\mu}_i^l = f^l\!\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}^l \, \mathbf{W}^l \mu_j^{l+1}\right)$$

L'erreur de prédiction devient :

$$\varepsilon_i^l = \mu_i^l - \hat{\mu}_i^l$$

### 2. Attention guidée par la surprise

Les scores d'attention sont calculés à partir des erreurs de prédiction :

$$e_{ij}^l = \text{LeakyReLU}\!\left((a^l)^\top [\varepsilon_i^l \,\|\, \varepsilon_j^l]\right)$$
$$\alpha_{ij}^l = \text{softmax}_j(e_{ij}^l)$$

### 3. Dynamique d'inférence (sans backprop global)

Les croyances $\mu_i^l$ sont mises à jour par descente de gradient locale sur l'énergie libre $\mathcal{F} = \sum_l \sum_i |\varepsilon_i^l|^2$ :

$$\dot{\mu}_i^l = -\varepsilon_i^l + \sum_{j \in \mathcal{N}(i)} \alpha_{ij}^{l-1} \cdot \left(\frac{\partial \hat{\mu}_j^{l-1}}{\partial \mu_i^l}\right)^\top \varepsilon_j^{l-1}$$

### 4. Apprentissage local (règle Hebbienne/PC)

Une fois les $\mu$ convergés, les poids $\mathbf{W}^l$ sont mis à jour sans rétropropagation globale :

$$\Delta \mathbf{W}^l \propto \sum_i \varepsilon_i^l \cdot \left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}^l \mu_j^{l+1}\right)^\top$$

---

```
Couche l+1   μ_{l+1}  ──(f^l, W^l)──►  μ̂_i^l   (prédiction top-down)
                                              │
                                              ▼
Couche l     μ_i^l   ────────────────►  ε_i^l = μ_i^l - μ̂_i^l
                ▲                             │
                │     correction locale        │  attention α_{ij} basée
                └──────────────────────────────┘  sur les erreurs
                         bottom-up
```

## Installation des dépendances

In [ ]:
%pip install torch>=2.0.0 matplotlib

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from typing import Optional, Tuple, List

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

print(f"PyTorch version: {torch.__version__}")

---
## Implémentation du modèle PC-GAT

### Couche PC-GAT (`PCGATLayer`)

Chaque couche maintient des **neurones de représentation** $\mu_i^l$ et des **neurones d'erreur** $\varepsilon_i^l$, et exécute une boucle d'inférence itérative avant toute mise à jour des poids.

In [ ]:
class PCGATLayer(nn.Module):
    """
    Single PC-GAT layer.

    Each node i maintains:
      - Representation neurons μᵢˡ  : current belief about the node state
      - Error neurons       εᵢˡ  : gap between belief and top-down prediction

    The layer runs an inference loop that iteratively:
      1. Computes top-down predictions using attention-weighted neighbor aggregation
      2. Computes prediction errors
      3. Updates attention scores based on those errors (surprise-guided)
      4. Updates beliefs via local gradient descent (no global backprop)

    After convergence, weights can be updated locally via `local_weight_update`.

    Args:
        in_features:        Dimensionality of the upper (input) layer representations.
        out_features:       Dimensionality of this layer's representations.
        n_inference_steps:  Number of iterative inference steps before convergence.
        inference_lr:       Step size for belief update dynamics.
        dropout:            Dropout rate applied to attention weights.
        activation:         Non-linearity used in the generative model ('relu', 'tanh', 'linear').
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        n_inference_steps: int = 20,
        inference_lr: float = 0.1,
        dropout: float = 0.0,
        activation: str = "relu",
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_inference_steps = n_inference_steps
        self.inference_lr = inference_lr

        # Wˡ : generative weight matrix for top-down predictions
        # Shape: [out_features, in_features]
        self.W = nn.Parameter(torch.empty(out_features, in_features))

        # aˡ : attention weight vector operating on concatenated error pairs
        # Shape: [2 * out_features]
        self.a = nn.Parameter(torch.empty(2 * out_features))

        self.dropout = nn.Dropout(p=dropout)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)

        if activation == "relu":
            self.activation = F.relu
        elif activation == "tanh":
            self.activation = torch.tanh
        elif activation == "linear":
            self.activation = lambda x: x
        else:
            raise ValueError(f"Unknown activation '{activation}'. Choose relu/tanh/linear.")

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.xavier_uniform_(self.W.data.unsqueeze(0))
        nn.init.xavier_uniform_(self.a.data.view(1, -1))

    # ------------------------------------------------------------------
    # Core computations
    # ------------------------------------------------------------------

    def compute_attention(
        self,
        errors: Tensor,
        edge_index: Tensor,
    ) -> Tensor:
        """
        Compute surprise-guided attention weights.

        eᵢⱼˡ = LeakyReLU(aˡᵀ · [εᵢˡ ‖ εⱼˡ])
        αᵢⱼˡ = softmax over neighbors j of node i
        """
        src, dst = edge_index

        e_concat = torch.cat([errors[src], errors[dst]], dim=-1)  # [E, 2*out_features]
        e_scores = self.leaky_relu(e_concat @ self.a)  # [E]
        e_scores = self.dropout(e_scores)

        n_nodes = errors.shape[0]
        alpha = self._sparse_softmax(e_scores, dst, n_nodes)
        return alpha

    def _sparse_softmax(self, scores: Tensor, dst: Tensor, n_nodes: int) -> Tensor:
        """Numerically stable softmax grouped by destination node."""
        max_scores = torch.full((n_nodes,), float("-inf"), device=scores.device)
        max_scores.scatter_reduce_(0, dst, scores, reduce="amax", include_self=True)
        max_scores = max_scores.nan_to_num(nan=0.0, posinf=0.0, neginf=0.0)

        shifted = scores - max_scores[dst]
        exp_scores = torch.exp(shifted)

        sum_exp = torch.zeros(n_nodes, device=scores.device)
        sum_exp.scatter_add_(0, dst, exp_scores)

        alpha = exp_scores / (sum_exp[dst] + 1e-8)
        return alpha

    def top_down_prediction(
        self,
        mu_upper: Tensor,
        alpha: Tensor,
        edge_index: Tensor,
    ) -> Tensor:
        """
        Generate top-down predictions via attention-weighted neighbor aggregation.

        μ̂ᵢˡ = fˡ(Σⱼ∈N(i) αᵢⱼˡ · Wˡ · μⱼˡ⁺¹)
        """
        src, dst = edge_index
        n_nodes = mu_upper.shape[0]

        transformed = mu_upper @ self.W.t()  # [N, out_features]
        messages = alpha.unsqueeze(-1) * transformed[src]  # [E, out_features]
        agg = torch.zeros(n_nodes, self.out_features, device=mu_upper.device)
        agg.scatter_add_(0, dst.unsqueeze(-1).expand_as(messages), messages)

        mu_hat = self.activation(agg)
        return mu_hat

    def inference_step(
        self,
        mu: Tensor,
        errors: Tensor,
        errors_lower: Optional[Tensor],
        alpha_lower: Optional[Tensor],
        edge_index: Tensor,
    ) -> Tensor:
        """
        One gradient-descent step on the variational free energy.

        μ̇ᵢˡ = -εᵢˡ  +  Σⱼ∈N(i) αᵢⱼˡ⁻¹ · (∂μ̂ⱼˡ⁻¹/∂μᵢˡ)ᵀ · εⱼˡ⁻¹
        """
        delta = -errors

        if errors_lower is not None and alpha_lower is not None:
            src, dst = edge_index
            weighted_err = alpha_lower.unsqueeze(-1) * errors_lower[dst]  # [E, lower_features]

            lower_features = errors_lower.shape[-1]
            backprop = torch.zeros(mu.shape[0], lower_features, device=mu.device)
            backprop.scatter_add_(0, src.unsqueeze(-1).expand_as(weighted_err), weighted_err)

            delta = delta + backprop @ self.W.t()  # [N, out_features]

        mu_new = mu + self.inference_lr * delta
        return mu_new

    def local_weight_update(
        self,
        errors: Tensor,
        mu_upper: Tensor,
        alpha: Tensor,
        edge_index: Tensor,
    ) -> None:
        """
        Accumulate the local Hebbian/PC weight gradient.

        ΔWˡ ∝ Σᵢ εᵢˡ · (Σⱼ∈N(i) αᵢⱼˡ · μⱼˡ⁺¹)ᵀ
        """
        src, dst = edge_index
        n_nodes = errors.shape[0]

        weighted_upper = alpha.unsqueeze(-1) * mu_upper[src]  # [E, in_features]
        agg_upper = torch.zeros(n_nodes, self.in_features, device=mu_upper.device)
        agg_upper.scatter_add_(0, dst.unsqueeze(-1).expand_as(weighted_upper), weighted_upper)

        delta_W = errors.t() @ agg_upper  # [out_features, in_features]

        if self.W.grad is None:
            self.W.grad = -delta_W.detach()
        else:
            self.W.grad.add_(-delta_W.detach())

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------

    def forward(
        self,
        mu_upper: Tensor,
        edge_index: Tensor,
        mu_init: Optional[Tensor] = None,
        errors_lower: Optional[Tensor] = None,
        alpha_lower: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Tensor, Tensor]:
        """
        Run iterative inference to convergence for this layer.

        Returns:
            mu:     Converged beliefs       [N, out_features]
            errors: Final prediction errors [N, out_features]
            alpha:  Final attention weights [E]
        """
        n_nodes = mu_upper.shape[0]

        mu = (
            mu_init.clone()
            if mu_init is not None
            else torch.zeros(n_nodes, self.out_features, device=mu_upper.device)
        )

        alpha = self._uniform_attention(edge_index, n_nodes, mu_upper.device)

        for _ in range(self.n_inference_steps):
            mu_hat = self.top_down_prediction(mu_upper, alpha, edge_index)
            errors = mu - mu_hat
            alpha = self.compute_attention(errors, edge_index)
            mu = self.inference_step(mu, errors, errors_lower, alpha_lower, edge_index)

        mu_hat = self.top_down_prediction(mu_upper, alpha, edge_index)
        errors = mu - mu_hat

        return mu, errors, alpha

    def _uniform_attention(
        self, edge_index: Tensor, n_nodes: int, device: torch.device
    ) -> Tensor:
        """Initialise with uniform (1/degree) attention weights."""
        src, dst = edge_index
        n_edges = edge_index.shape[1]
        degree = torch.zeros(n_nodes, device=device)
        degree.scatter_add_(0, dst, torch.ones(n_edges, device=device))
        alpha = 1.0 / (degree[dst] + 1e-8)
        return alpha

### Modèle PC-GAT complet (`PCGAT`)

Le modèle complet est une pile de couches `PCGATLayer`. L'entrée $x$ est traitée comme la représentation de plus haut niveau (couche $L$) et chaque couche inférieure infère ses croyances par rapport aux prédictions de la couche supérieure.

In [ ]:
class PCGAT(nn.Module):
    """
    Full PC-GAT model: a stack of PCGATLayer modules.

    Args:
        layer_dims:         List of feature dimensions from input to output.
                            E.g. [128, 64, 32] → 2-layer PC-GAT,
                            input dim 128, hidden 64, output 32.
        n_inference_steps:  Inference iterations per layer per forward pass.
        inference_lr:       Step size for belief updates.
        dropout:            Attention dropout rate.
        activation:         Generative activation ('relu', 'tanh', 'linear').
    """

    def __init__(
        self,
        layer_dims: List[int],
        n_inference_steps: int = 20,
        inference_lr: float = 0.1,
        dropout: float = 0.0,
        activation: str = "relu",
    ):
        super().__init__()

        if len(layer_dims) < 2:
            raise ValueError("layer_dims must have at least 2 elements.")

        self.layer_dims = layer_dims
        self.n_layers = len(layer_dims) - 1

        self.layers = nn.ModuleList(
            [
                PCGATLayer(
                    in_features=layer_dims[l + 1],
                    out_features=layer_dims[l],
                    n_inference_steps=n_inference_steps,
                    inference_lr=inference_lr,
                    dropout=dropout,
                    activation=activation,
                )
                for l in range(self.n_layers)
            ]
        )

    def forward(
        self,
        x: Tensor,
        edge_index: Tensor,
    ) -> Tuple[Tensor, List[Tensor], List[Tensor]]:
        """
        Run inference through all PC-GAT layers.

        Returns:
            output:     Beliefs at the bottom layer      [N, layer_dims[0]]
            all_errors: List of prediction errors per layer (bottom → top)
            all_alpha:  List of attention weights per layer (bottom → top)
        """
        all_mu: List[Tensor] = []
        all_errors: List[Tensor] = []
        all_alpha: List[Tensor] = []

        mu_upper = x
        errors_lower: Optional[Tensor] = None
        alpha_lower: Optional[Tensor] = None

        for layer in reversed(self.layers):
            mu, errors, alpha = layer(
                mu_upper=mu_upper,
                edge_index=edge_index,
                errors_lower=errors_lower,
                alpha_lower=alpha_lower,
            )
            all_mu.append(mu)
            all_errors.append(errors)
            all_alpha.append(alpha)

            mu_upper = mu
            errors_lower = errors
            alpha_lower = alpha

        return mu, all_errors, all_alpha

    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------

    def total_free_energy(self, all_errors: List[Tensor]) -> Tensor:
        """Variational free energy: F = Σₗ Σᵢ |εᵢˡ|²"""
        return sum(e.pow(2).sum() for e in all_errors)

    def anomaly_scores(self, all_errors: List[Tensor]) -> Tensor:
        """
        Per-node anomaly score = sum of squared prediction errors across layers.

        Returns:
            scores: Anomaly score per node [N]
        """
        return sum(e.pow(2).mean(dim=-1) for e in all_errors)

    def local_update_all(
        self,
        all_errors: List[Tensor],
        x: Tensor,
        all_mu: List[Tensor],
        all_alpha: List[Tensor],
        edge_index: Tensor,
    ) -> None:
        """
        Accumulate local Hebbian weight gradients for all layers.
        Call this after `forward`, then call `optimizer.step()` to apply.
        """
        for l, layer in enumerate(reversed(self.layers)):
            mu_upper = x if l == 0 else all_mu[l - 1]
            layer.local_weight_update(
                errors=all_errors[l],
                mu_upper=mu_upper,
                alpha=all_alpha[l],
                edge_index=edge_index,
            )

print("✅ Classes PCGATLayer et PCGAT définies.")

---
## Génération du graphe synthétique

On crée un graphe aléatoire de type Barabási–Albert avec des étiquettes de nœuds binaires.
Les features des nœuds sont discriminatives : la classe 1 est décalée de +1 en moyenne.

In [ ]:
def make_synthetic_graph(
    n_nodes: int = 200,
    feat_dim: int = 16,
    n_edges_per_node: int = 3,
    seed: int = 42,
) -> tuple:
    """
    Create a random Barabási–Albert graph with binary node labels.

    Returns:
        x:          Node features  [N, feat_dim]
        edge_index: Edges          [2, E]  (undirected, stored as both directions)
        labels:     Node labels    [N]  (0 or 1)
    """
    torch.manual_seed(seed)

    x = torch.randn(n_nodes, feat_dim)

    labels = torch.zeros(n_nodes, dtype=torch.long)
    labels[n_nodes // 2 :] = 1

    x[labels == 1] += 1.0

    edges = []
    for i in range(n_nodes):
        edges.append((i, (i + 1) % n_nodes))
        edges.append(((i + 1) % n_nodes, i))
        for _ in range(n_edges_per_node - 1):
            j = torch.randint(n_nodes, (1,)).item()
            if j != i:
                edges.append((i, j))
                edges.append((j, i))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return x, edge_index, labels


# Génération
N_NODES = 200
FEAT_DIM = 16
LAYER_DIMS = [8, 16, FEAT_DIM]   # output=8, hidden=16, input=FEAT_DIM
N_CLASSES = 2

x, edge_index, labels = make_synthetic_graph(n_nodes=N_NODES, feat_dim=FEAT_DIM)

print(f"Graphe : {N_NODES} nœuds, {edge_index.shape[1]} arêtes, {FEAT_DIM} features par nœud")
print(f"Classe 0 : {(labels == 0).sum().item()} nœuds | Classe 1 : {(labels == 1).sum().item()} nœuds")

In [ ]:
# Visualisation des features du graphe (2 premières dimensions)
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["steelblue" if l == 0 else "tomato" for l in labels.tolist()]
ax.scatter(x[:, 0].numpy(), x[:, 1].numpy(), c=colors, alpha=0.7, edgecolors="k", linewidths=0.4)
ax.set_xlabel("Feature dim 0")
ax.set_ylabel("Feature dim 1")
ax.set_title("Features des nœuds (2 premières dimensions)\nBleu = Classe 0  |  Rouge = Classe 1")
plt.tight_layout()
plt.show()

---
## Mode 1 – Apprentissage PC local (sans backprop global)

Les poids sont mis à jour uniquement via la règle de Hebb/PC — **aucun appel à `loss.backward()`**.
Le modèle minimise l'énergie libre variationnelle $\mathcal{F}$ par règles locales.

In [ ]:
def train_local_pc(
    model: PCGAT,
    x: torch.Tensor,
    edge_index: torch.Tensor,
    labels: torch.Tensor,
    n_epochs: int = 50,
    lr: float = 1e-3,
) -> List[float]:
    """
    Train PC-GAT with purely local Hebbian weight updates.
    Returns the free-energy history for plotting.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []

    print("=== Mode 1 : Apprentissage PC local (sans backprop global) ===")
    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()

        output, all_errors, all_alpha = model(x, edge_index)

        # Reconstruct per-layer beliefs
        all_mu = []
        mu_upper = x
        for layer in reversed(model.layers):
            mu, errors, alpha = layer(mu_upper=mu_upper, edge_index=edge_index)
            all_mu.append(mu)
            mu_upper = mu

        model.local_update_all(all_errors, x, all_mu, all_alpha, edge_index)
        optimizer.step()

        fe = model.total_free_energy(all_errors).item()
        history.append(fe)

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:4d}  |  Énergie libre: {fe:.4f}")

    print("Apprentissage PC local terminé.\n")
    return history


pc_model = PCGAT(
    layer_dims=LAYER_DIMS,
    n_inference_steps=10,
    inference_lr=0.05,
    dropout=0.1,
)

history_local = train_local_pc(pc_model, x, edge_index, labels, n_epochs=50)

In [ ]:
# Courbe d'énergie libre
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history_local) + 1), history_local, color="steelblue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Énergie libre $\mathcal{F}$")
plt.title("Mode 1 – Décroissance de l'énergie libre (apprentissage local PC)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Mode 2 – Apprentissage hybride (backbone PC + tête de classification backprop)

Le backbone PC-GAT produit des représentations de nœuds, et une tête linéaire entraînée par backprop classique effectue la classification. C'est la stratégie hybride décrite dans la littérature PC.

In [ ]:
class PCGATClassifier(nn.Module):
    """PC-GAT backbone with a linear classification head."""

    def __init__(self, layer_dims: list, n_classes: int, **kwargs):
        super().__init__()
        self.backbone = PCGAT(layer_dims, **kwargs)
        self.head = nn.Linear(layer_dims[0], n_classes)

    def forward(self, x, edge_index):
        output, all_errors, all_alpha = self.backbone(x, edge_index)
        logits = self.head(output)
        return logits, all_errors, all_alpha


def train_hybrid(
    model: PCGATClassifier,
    x: torch.Tensor,
    edge_index: torch.Tensor,
    labels: torch.Tensor,
    n_epochs: int = 100,
    lr: float = 1e-3,
) -> Tuple[List[float], List[float], List[float]]:
    """
    Hybrid training: PC-GAT backbone (local rules) + backprop on the task head.
    Returns (loss_history, acc_history, fe_history).
    """
    head_optimizer = optim.Adam(model.head.parameters(), lr=lr)
    backbone_optimizer = optim.Adam(model.backbone.parameters(), lr=lr * 0.1)
    criterion = nn.CrossEntropyLoss()

    loss_history, acc_history, fe_history = [], [], []

    print("=== Mode 2 : Hybride (backbone PC + tête backprop) ===")
    for epoch in range(1, n_epochs + 1):
        model.train()
        head_optimizer.zero_grad()
        backbone_optimizer.zero_grad()

        logits, all_errors, all_alpha = model(x, edge_index)

        loss = criterion(logits, labels)
        loss.backward()
        head_optimizer.step()

        fe = model.backbone.total_free_energy(all_errors).item()
        acc = (logits.argmax(dim=-1) == labels).float().mean().item()

        loss_history.append(loss.item())
        acc_history.append(acc)
        fe_history.append(fe)

        if epoch % 20 == 0:
            print(
                f"  Epoch {epoch:4d}  |  Loss: {loss.item():.4f}  "
                f"|  Acc: {acc:.3f}  |  Énergie libre: {fe:.4f}"
            )

    print("Apprentissage hybride terminé.\n")
    return loss_history, acc_history, fe_history


hybrid_model = PCGATClassifier(
    layer_dims=LAYER_DIMS,
    n_classes=N_CLASSES,
    n_inference_steps=10,
    inference_lr=0.05,
    dropout=0.1,
)

loss_hist, acc_hist, fe_hist = train_hybrid(hybrid_model, x, edge_index, labels, n_epochs=100)

In [ ]:
# Visualisation des métriques d'entraînement hybride
epochs = range(1, len(loss_hist) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, loss_hist, color="tomato", linewidth=2)
axes[0].set_title("Perte (Cross-Entropy)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, acc_hist, color="seagreen", linewidth=2)
axes[1].set_title("Précision de classification")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

axes[2].plot(epochs, fe_hist, color="orchid", linewidth=2)
axes[2].set_title("Énergie libre $\mathcal{F}$")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Free energy")
axes[2].grid(alpha=0.3)

fig.suptitle("Mode 2 – Apprentissage hybride PC-GAT", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Précision finale : {acc_hist[-1]:.1%}")

---
## Détection d'anomalies native

Une des propriétés remarquables de PC-GAT est sa capacité de **détection d'anomalies sans objectif supplémentaire**.
Les nœuds atypiques (features hors distribution) génèrent des erreurs de prédiction $\varepsilon$ élevées, ce qui se traduit directement par un score d'anomalie élevé.

In [ ]:
def demo_anomaly_detection(
    model: PCGAT,
    x: torch.Tensor,
    edge_index: torch.Tensor,
    n_anomalies: int = 10,
) -> Tuple[torch.Tensor, set, set]:
    """
    Inject anomalous nodes (random features) and measure anomaly scores.
    Returns (scores, injected_set, detected_set).
    """
    model.eval()
    x_noisy = x.clone()
    anomaly_idx = torch.randperm(x.shape[0])[:n_anomalies]
    x_noisy[anomaly_idx] = torch.randn(n_anomalies, x.shape[1]) * 5.0  # outliers

    with torch.no_grad():
        _, all_errors, _ = model(x_noisy, edge_index)
        scores = model.anomaly_scores(all_errors)

    top_k = scores.topk(n_anomalies).indices
    injected = set(anomaly_idx.tolist())
    detected = set(top_k.tolist())
    overlap = len(injected & detected)

    print("=== Démonstration de détection d'anomalies ===")
    print(f"  Nœuds anomaliques injectés : {sorted(injected)}")
    print(f"  Top-{n_anomalies} par erreur PC    : {sorted(detected)}")
    print(f"  Rappel (recall)            : {overlap}/{n_anomalies}  ({overlap/n_anomalies:.0%})")

    return scores, injected, detected


N_ANOMALIES = 10
scores, injected, detected = demo_anomaly_detection(pc_model, x, edge_index, n_anomalies=N_ANOMALIES)

In [ ]:
# Visualisation des scores d'anomalie
scores_np = scores.detach().numpy()
node_ids = np.arange(N_NODES)

fig, ax = plt.subplots(figsize=(12, 4))

bar_colors = []
for i in node_ids:
    if i in injected and i in detected:
        bar_colors.append("tomato")       # vrai positif
    elif i in injected:
        bar_colors.append("orange")       # faux négatif
    elif i in detected:
        bar_colors.append("gold")         # faux positif
    else:
        bar_colors.append("steelblue")    # vrai négatif

ax.bar(node_ids, scores_np, color=bar_colors, width=1.0, edgecolor="none")
ax.set_xlabel("Indice du nœud")
ax.set_ylabel("Score d'anomalie (erreur PC²)")
ax.set_title("Scores d'anomalie par nœud")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="tomato",    label="Vrai positif (injecté & détecté)"),
    Patch(facecolor="orange",    label="Faux négatif (injecté, non détecté)"),
    Patch(facecolor="gold",      label="Faux positif (détecté, non injecté)"),
    Patch(facecolor="steelblue", label="Vrai négatif"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Visualisation des représentations apprises

Comparaison des représentations produites par le modèle après entraînement pour les deux classes de nœuds.

In [ ]:
pc_model.eval()
with torch.no_grad():
    output, all_errors, _ = pc_model(x, edge_index)

# output shape: [N, layer_dims[0]] = [200, 8]
repr_np = output.numpy()
labels_np = labels.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Scatter des 2 premières dimensions des représentations ---
ax = axes[0]
colors = np.where(labels_np == 0, "steelblue", "tomato")
ax.scatter(repr_np[:, 0], repr_np[:, 1], c=colors, alpha=0.75, edgecolors="k", linewidths=0.3)
ax.set_xlabel("Dim 0")
ax.set_ylabel("Dim 1")
ax.set_title("Représentations apprises (couche de sortie)\nBleu = Classe 0  |  Rouge = Classe 1")

# --- Distribution des erreurs de prédiction par couche ---
ax2 = axes[1]
for l_idx, errors in enumerate(all_errors):
    err_norm = errors.pow(2).mean(dim=-1).detach().numpy()
    ax2.hist(err_norm, bins=30, alpha=0.6, label=f"Couche {l_idx}")
ax2.set_xlabel("Erreur quadratique moyenne par nœud")
ax2.set_ylabel("Nombre de nœuds")
ax2.set_title("Distribution des erreurs de prédiction par couche")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Récapitulatif

| Composant | Description |
|---|---|
| `PCGATLayer` | Couche unique avec boucle d'inférence itérative et attention guidée par la surprise |
| `PCGAT` | Modèle complet empilant plusieurs `PCGATLayer` |
| `PCGATClassifier` | Backbone PC-GAT + tête linéaire pour la classification |
| `total_free_energy` | $\mathcal{F} = \sum_l \sum_i \|\varepsilon_i^l\|^2$ — métrique de qualité du modèle |
| `anomaly_scores` | Score d'anomalie par nœud via les erreurs de prédiction |

### Points forts de PC-GAT

1. **Apprentissage local** : pas de backpropagation globale nécessaire pour les couches internes
2. **Attention adaptative** : les poids d'attention évoluent au fil de l'inférence (le réseau "réfléchit")
3. **Détection d'anomalies gratuite** : les nœuds hors distribution sont automatiquement identifiés par leurs erreurs élevées
4. **Flexibilité** : compatible avec un entraînement hybride (tête backprop) pour les tâches supervisées